# Tests d'hypothèses — World Happiness Report (2015–2019)

Chaque test est accompagné de son code, ses résultats et son interprétation complète.

In [17]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.linear_model import LinearRegression

In [2]:
df = pd.read_csv('World_happiness_clean.csv', index_col=0)

In [3]:
df.head()

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,Year
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738,2015
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201,2015
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204,2015
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531,2015
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176,2015


In [4]:
df = df.reset_index(drop=True)

In [5]:
df.head()

,Country,Region,Happiness Rank,Happiness Score,Standard Error,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption),Generosity,Dystopia Residual,Year
0,Switzerland,Western Europe,1,7.587,0.03411,1.39651,1.34951,0.94143,0.66557,0.41978,0.29678,2.51738,2015
1,Iceland,Western Europe,2,7.561,0.04884,1.30232,1.40223,0.94784,0.62877,0.14145,0.43630,2.70201,2015
2,Denmark,Western Europe,3,7.527,0.03328,1.32548,1.36058,0.87464,0.64938,0.48357,0.34139,2.49204,2015
3,Norway,Western Europe,4,7.522,0.03880,1.45900,1.33095,0.88521,0.66973,0.36503,0.34699,2.46531,2015
4,Canada,North America,5,7.427,0.03553,1.32629,1.32261,0.90563,0.63297,0.32957,0.45811,2.45176,2015


---
## 1. Test de normalité — Shapiro-Wilk

**Hypothèse :** La distribution du Happiness Score suit-elle une loi normale ?

In [8]:
sample = df['Happiness Score'].sample(200, random_state=42)
stat, p = stats.shapiro(sample)

print(f"Statistique W : {round(stat, 4)}")
print(f"p-value       : {round(p, 4)}")
print()
if p < 0.05:
    print("Conclusion : p < 0.05 alors on rejette l'hypothèse de normalité.")
    print("La distribution n'est pas parfaitement normale.")
    print("En pratique, l'écart est faible (W proche de 1), mais pour être")
    print("rigoureux il vaut mieux utiliser des tests non-paramétriques")
    print("ou s'appuyer sur le théorème central limite (n > 30).")
else:
    print("Conclusion : p >= 0.05 alors on ne peut pas rejeter la normalité.")

Statistique W : 0.982
p-value       : 0.0115

Conclusion : p < 0.05 alors on rejette l'hypothèse de normalité.
La distribution n'est pas parfaitement normale.
En pratique, l'écart est faible (W proche de 1), mais pour être
rigoureux il vaut mieux utiliser des tests non-paramétriques
ou s'appuyer sur le théorème central limite (n > 30).


---
## 2. T-test — Europe de l'Ouest vs Afrique subsaharienne

**Hypothèse :** Les deux régions ont-elles des scores significativement différents ?

In [11]:
europe  = df[df['Region'] == 'Western Europe']['Happiness Score']
afrique = df[df['Region'] == 'Sub-Saharan Africa']['Happiness Score']

stat, p = stats.ttest_ind(europe, afrique)

print(f"Moyenne Europe de l'Ouest       : {round(europe.mean(), 4)}")
print(f"Moyenne Afrique subsaharienne   : {round(afrique.mean(), 4)}")
print(f"Différence de score             : {round(europe.mean() - afrique.mean(), 4)}")
print(f"Statistique t                   : {round(stat, 4)}")
print(f"p-value                         : {p:.2e}")
print()
if p < 0.05:
    print("Conclusion : p < 0.05 alors la différence est statistiquement significative.")
    print(f"Une différence de {round(europe.mean() - afrique.mean(), 4)} points n'est pas due au hasard.")
    print("Ce résultat confirme ce que l'on voit sur les graphiques.")

Moyenne Europe de l'Ouest       : 6.7593
Moyenne Afrique subsaharienne   : 4.1885
Différence de score             : 2.5708
Statistique t                   : 32.2646
p-value                         : 6.36e-99

Conclusion : p < 0.05 alors la différence est statistiquement significative.
Une différence de 2.5708 points n'est pas due au hasard.
Ce résultat confirme ce que l'on voit sur les graphiques.


---
## 3. ANOVA — Comparaison de toutes les régions

**Hypothèse :** Le score de bonheur est-il différent entre toutes les régions géographiques ?

In [12]:
groupes = [g['Happiness Score'].values for _, g in df.groupby('Region')]
stat, p = stats.f_oneway(*groupes)

print(f"Statistique F : {round(stat, 4)}")
print(f"p-value       : {p:.2e}")
print()

moyennes = (df.groupby('Region')['Happiness Score']
            .mean()
            .sort_values(ascending=False)
            .round(3))
print("Scores moyens par région :")
print(moyennes.to_string())
print()
if p < 0.05:
    print("Conclusion : p < 0.05 alors il existe au moins une paire de régions")
    print("significativement différente. La région géographique est fortement")
    print("associée au niveau de bonheur.")
    print("Un test post-hoc de Tukey (voir ci-dessous) identifie lesquelles.")

Statistique F : 123.6193
p-value       : 2.43e-152

Scores moyens par région :
Region
Australia and New Zealand          7.295
North America                      7.175
Western Europe                     6.759
Latin America and Caribbean        6.021
Unknown                            5.947
Eastern Asia                       5.630
Central and Eastern Europe         5.429
Middle East and Northern Africa    5.337
Southeastern Asia                  5.335
Southern Asia                      4.581
Sub-Saharan Africa                 4.188

Conclusion : p < 0.05 alors il existe au moins une paire de régions
significativement différente. La région géographique est fortement
associée au niveau de bonheur.
Un test post-hoc de Tukey (voir ci-dessous) identifie lesquelles.


### Test post-hoc de Tukey

L'ANOVA dit qu'il y a une différence quelque part. Tukey dit exactement où.

In [13]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

tukey = pairwise_tukeyhsd(df['Happiness Score'], df['Region'], alpha=0.05)
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                         columns=tukey._results_table.data[0])
sig    = tukey_df[tukey_df['reject'] == True]
non_sig = tukey_df[tukey_df['reject'] == False]

print(f"Paires testées               : {len(tukey_df)}")
print(f"Paires significativement diff: {len(sig)}")
print(f"Paires non significatives    : {len(non_sig)}")
print()
print("Les paires non significatives correspondent à des régions")
print("avec des scores proches (ex : Moyen-Orient et Asie du Sud-Est).")
print()
print("Extrait des paires les plus différentes :")
print(sig[['group1','group2','meandiff','p-adj']]
      .sort_values('meandiff')
      .head(5)
      .to_string(index=False))

Paires testées               : 55
Paires significativement diff: 35
Paires non significatives    : 20

Les paires non significatives correspondent à des régions
avec des scores proches (ex : Moyen-Orient et Asie du Sud-Est).

Extrait des paires les plus différentes :
                   group1             group2  meandiff  p-adj
Australia and New Zealand Sub-Saharan Africa   -3.1061    0.0
            North America Sub-Saharan Africa   -2.9862    0.0
Australia and New Zealand      Southern Asia   -2.7139    0.0
            North America      Southern Asia   -2.5940    0.0
Australia and New Zealand  Southeastern Asia   -1.9594    0.0


---
## 4. Corrélations — Quels facteurs sont liés au bonheur ?

**Hypothèse :** Chacun des 6 facteurs est-il corrélé avec le Happiness Score ?

In [14]:
facteurs = [
    'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)',
    'Freedom', 'Trust (Government Corruption)', 'Generosity'
]

In [15]:
print(f"{'Facteur':<35} {'r':>8} {'p-value':>12}  Interprétation")
print("-" * 80)

interpretations = {
    (0.7, 1.0)  : "Corrélation forte",
    (0.5, 0.7)  : "Corrélation modérée à forte",
    (0.3, 0.5)  : "Corrélation faible à modérée",
    (0.0, 0.3)  : "Corrélation très faible",
}

for factor in facteurs:
    r, p = stats.pearsonr(df[factor], df['Happiness Score'])
    for (low, high), label in interpretations.items():
        if low <= abs(r) < high:
            interp = label
            break
    print(f"{factor:<35} {r:>8.3f} {p:>12.2e}  {interp}")

print()
print("Note : tous les p-values sont < 0.05, donc toutes les corrélations sont")
print("statistiquement significatives. Mais r = 0.139 pour la générosité")
print("signifie que l'effet est négligeable en pratique — c'est la taille")
print("de l'échantillon (776 lignes) qui rend le test significatif.")

Facteur                                    r      p-value  Interprétation
--------------------------------------------------------------------------------
Economy (GDP per Capita)               0.789    6.03e-166  Corrélation forte
Family                                 0.648     1.05e-93  Corrélation modérée à forte
Health (Life Expectancy)               0.744    9.69e-138  Corrélation forte
Freedom                                0.551     1.01e-62  Corrélation modérée à forte
Trust (Government Corruption)          0.401     2.90e-31  Corrélation faible à modérée
Generosity                             0.139     1.02e-04  Corrélation très faible

Note : tous les p-values sont < 0.05, donc toutes les corrélations sont
statistiquement significatives. Mais r = 0.139 pour la générosité
signifie que l'effet est négligeable en pratique — c'est la taille
de l'échantillon (776 lignes) qui rend le test significatif.


---
## 5. Tendance temporelle

**Hypothèse :** Le bonheur mondial a-t-il augmenté ou diminué entre 2015 et 2019 ?

In [16]:
score_annee = df.groupby('Year')['Happiness Score'].mean()
r, p = stats.pearsonr(score_annee.index, score_annee.values)

print("Score moyen par année :")
for yr, sc in score_annee.items():
    print(f"  {yr} : {round(sc, 4)}")

print()
print(f"Corrélation temps-score : r = {round(r, 4)}")
print(f"p-value                 : {round(p, 4)}")
print()
if p >= 0.05:
    print("Conclusion : p >= 0.05 alors aucune tendance significative détectée.")
    print("Avec seulement 5 points de données, il est impossible de conclure")
    print("à une hausse ou une baisse fiable. Les variations observées")
    print(f"({round(score_annee.max() - score_annee.min(), 4)} pts sur 5 ans) sont")
    print("trop faibles pour être distinguées du bruit statistique.")
    print("Il faudrait des données sur une plus longue période.")

Score moyen par année :
  2015 : 5.3757
  2016 : 5.3822
  2017 : 5.354
  2018 : 5.3676
  2019 : 5.4066

Corrélation temps-score : r = 0.3829
p-value                 : 0.5247

Conclusion : p >= 0.05 alors aucune tendance significative détectée.
Avec seulement 5 points de données, il est impossible de conclure
à une hausse ou une baisse fiable. Les variations observées
(0.0526 pts sur 5 ans) sont
trop faibles pour être distinguées du bruit statistique.
Il faudrait des données sur une plus longue période.


---
## 6. Régression linéaire multiple

**Hypothèse :** Les 6 facteurs combinés expliquent-ils le Happiness Score ? Lequel contribue le plus ?

In [18]:
X = df[facteurs].values
y = df['Happiness Score'].values
model = LinearRegression().fit(X, y)

r2 = model.score(X, y)
print(f"R² = {round(r2, 4)} alors le modèle explique {round(r2*100, 4)}% de la variance du score")
print()
print(f"{'Facteur':<35} {'Coefficient':>12}")
print("-" * 50)
coefs = sorted(zip(facteurs, model.coef_), key=lambda x: abs(x[1]), reverse=True)
for name, coef in coefs:
    print(f"{name:<35} {coef:>12.4f}")

print()
print("Interprétation des coefficients :")
print("Chaque coefficient indique de combien augmente le score si ce facteur")
print("augmente d'1 unité, toutes choses égales par ailleurs.")
print()
print("La liberté (Freedom) a le coefficient le plus élevé (1.475),")
print("ce qui signifie que quand on contrôle les autres facteurs,")
print("c'est le levier le plus fort — devant le PIB (1.143).")
print()
print(f"Les {round((1-r2)*100, 4)}% restants correspondent à des facteurs non capturés")
print("par ce dataset (qualité des institutions, culture, etc.).")

R² = 0.7642 alors le modèle explique 76.4165% de la variance du score

Facteur                              Coefficient
--------------------------------------------------
Freedom                                   1.4749
Economy (GDP per Capita)                  1.1432
Health (Life Expectancy)                  1.0215
Trust (Government Corruption)             0.8526
Family                                    0.6388
Generosity                                0.5910

Interprétation des coefficients :
Chaque coefficient indique de combien augmente le score si ce facteur
augmente d'1 unité, toutes choses égales par ailleurs.

La liberté (Freedom) a le coefficient le plus élevé (1.475),
ce qui signifie que quand on contrôle les autres facteurs,
c'est le levier le plus fort — devant le PIB (1.143).

Les 23.5835% restants correspondent à des facteurs non capturés
par ce dataset (qualité des institutions, culture, etc.).


---
## 7. Test du Chi-2 — La région influence-t-elle le fait d'être dans le Top 50 ?

**Hypothèse :** La région géographique et le classement Top 50 sont-ils indépendants ?

In [19]:
df['Top50'] = (df['Happiness Rank'] <= 50).astype(int)
table = pd.crosstab(df['Region'], df['Top50'])
table.columns = ['Hors Top 50', 'Dans le Top 50']

stat, p, dof, expected = stats.chi2_contingency(table)

print(f"Statistique Chi-2 : {round(stat, 4)}")
print(f"p-value           : {p:.2e}")
print(f"Degrés de liberté : {dof}")
print()
print("Table de contingence :")
print(table.to_string())
print()
if p < 0.05:
    print("Conclusion : p < 0.05 alors la région et le classement Top 50 ne sont pas")
    print("indépendants. Il existe une association très forte entre les deux.")
    print()
    print("En clair :")
    print("  - Aucun pays d'Afrique subsaharienne ou d'Asie du Sud ne figure")
    print("    dans le Top 50 sur toute la période 2015–2019.")
    print("  - L'Europe de l'Ouest y est représentée à 83.5% (86/103 occurrences).")
    print("  - Ce résultat soulève des questions sur les inégalités structurelles")
    print("    mondiales qui vont bien au-delà du simple classement.")

Statistique Chi-2 : 340.4852
p-value           : 4.16e-67
Degrés de liberté : 10

Table de contingence :
                                 Hors Top 50  Dans le Top 50
Region                                                      
Australia and New Zealand                  0              10
Central and Eastern Europe               120              24
Eastern Asia                              22               6
Latin America and Caribbean               41              68
Middle East and Northern Africa           65              31
North America                              0              10
Southeastern Asia                         32              12
Southern Asia                             35               0
Sub-Saharan Africa                       195               0
Unknown                                    1               1
Western Europe                            17              86

Conclusion : p < 0.05 alors la région et le classement Top 50 ne sont pas
indépendants. Il existe une

---
## Résumé des conclusions

| Test | Résultat clé | Conclusion |
|------|-------------|------------|
| Normalité (Shapiro-Wilk) | p = 0.011 | Distribution non-normale |
| T-test Europe vs Afrique | p ≈ 0 | Différence réelle de +2.57 pts |
| ANOVA toutes régions | F = 123.6 | 35 paires sur 55 significativement différentes |
| Corrélation GDP | r = 0.789 | Lien fort — facteur le plus corrélé |
| Corrélation Generosity | r = 0.139 | Lien négligeable en pratique |
| Tendance temporelle | p = 0.525 | Aucune tendance sur 2015–2019 |
| Régression multiple | R² = 0.764 | 6 facteurs expliquent 76.4% du score |
| Chi-2 Région vs Top 50 | p ≈ 0 | La région détermine fortement le classement |
